This notebook is for data loading/preparation and cleaning

In [1]:
import pandas as pd

In [ ]:
# =========================
# 1) load data
# =========================
loads = pd.read_csv("raw_datasets/loads.csv")
routes = pd.read_csv("raw_datasets/routes.csv")
trips = pd.read_csv("raw_datasets/trips.csv")
fuel = pd.read_csv("raw_datasets/fuel_purchases.csv")
customers = pd.read_csv("raw_datasets/customers.csv")  # optional

In [ ]:
"""
Notes Arne:
The load type colum consists only of dry or refrigerated

We could use the primary freight type instead and just assume that this was the carried load (it is not 100% correct, but maybe it can serve as a good starting point)
"""

# =========================
# 2) define constants for column names
# =========================
# LOADS
LOAD_ID = "load_id"
LOAD_ROUTE_ID = "route_id"
LOAD_CUSTOMER_ID = "customer_id"
LOAD_REVENUE = "revenue"
LOAD_BOOKING_TYPE = "load_type"
LOAD_WEIGHT_LBS = "weight_lbs"
LOAD_NUMBER_OF_PIECES = "pieces"
LOAD_FUEL_SURCHARGE = "fuel_surcharge" 


# ROUTES
ROUTE_ID = "route_id"
ROUTE_DISTANCE_MILES = "typical_distance_miles"  # z.B. distance_miles, miles, route_distance
ROUTE_ORIGIN = "origin_city"
ROUTE_DEST = "destination_city"
ROUTE_BASE_RATE_PER_MILE = "base_rate_per_mile"  # optional, z.B. base_rate_per_mile, typical_rate_per_mile
ROUTE_FUEL_SURCHARGE_PER_MILE = "fuel_surcharge_rate"  # optional, z.B. fuel_surcharge_per_mile, typical_fuel_surcharge_per_mile

# TRIPS
TRIP_ID = "trip_id"
TRIP_LOAD_ID = "load_id"
TRIP_DURATION_HOURS = "actual_duration_hours"  # z.B. duration_hours
TRIP_FUEL_CONSUMPTION = "fuel_consumption"  # optional

# FUEL_PURCHASES
FUEL_TRIP_ID = "trip_id"
FUEL_PRICE_PER_GALLON = "price_per_gallon"  # z.B. fuel_price, price_per_gallon
FUEL_QUANTITY_GALLONS = "gallons"  # z.B. gallons
FUEL_TOTAL_COST = "total_cost"

# CUSTOMERS
CUSTOMER_ID = "customer_id"
CUSTOMER_PRIMARY_FREIGHT_TYPE = "primary_freight_type"  # optional
CUSTOMER_ANNUAL_REVENUE_POTENTIAL = "annual_revenue_potential"  # optional

In [ ]:
# =========================
# 3) preprocess fuel_purchases to get trip-level fuel features
# =========================
"""
Notes Arne for the fuel_purchases table:
 - There are multiple fuel purchases per trip, so we need to aggregate them to the trip level.
 - we need to calculate the average gas_price_per_gallon, total quantity of gallons, and total cost for each trip.
"""

fuel_agg = fuel.groupby(TRIP_ID, as_index=False).agg(
    avg_fuel_price=(FUEL_PRICE_PER_GALLON, "mean"),
    total_fuel_quantity=(FUEL_QUANTITY_GALLONS, "sum"),
    total_fuel_cost=(FUEL_TOTAL_COST, "sum"),
)

fuel_agg.head()

,trip_id,avg_fuel_price,total_fuel_quantity,total_fuel_cost
0,TRIP00000001,3.845000,201.2,779.45
1,TRIP00000002,4.008000,187.8,752.70
2,TRIP00000003,4.041167,928.5,3796.39
3,TRIP00000005,3.682000,139.2,512.53
4,TRIP00000006,4.247400,771.5,3311.07


In [5]:
# =========================
# 4) TRIPS + FUEL MERGEN
# =========================
trips_with_fuel = trips.merge(fuel_agg, left_on=TRIP_ID, right_on=TRIP_ID, how="left")

In [6]:
#trips.head()
trips_with_fuel.head()

,trip_id,load_id,driver_id,truck_id,trailer_id,dispatch_date,actual_distance_miles,actual_duration_hours,fuel_gallons_used,average_mpg,idle_time_hours,trip_status,avg_fuel_price,total_fuel_quantity,total_fuel_cost
0,TRIP00000001,LOAD00000001,DRV00117,TRK00035,TRL00167,2022-01-01,1314,26.2,183.8,7.15,3.5,Completed,3.845000,201.2,779.45
1,TRIP00000002,LOAD00000002,DRV00141,TRK00108,TRL00082,2022-01-01,515,8.6,93.6,5.50,8.3,Completed,4.008000,187.8,752.70
2,TRIP00000003,LOAD00000003,DRV00032,TRK00031,TRL00138,2022-01-01,2509,45.0,339.1,7.40,12.0,Completed,4.041167,928.5,3796.39
3,TRIP00000004,LOAD00000004,DRV00083,TRK00105,TRL00018,2022-01-01,717,11.1,110.3,6.50,9.6,Completed,NaN,NaN,NaN
4,TRIP00000005,LOAD00000005,DRV00044,TRK00076,TRL00054,2022-01-01,2243,35.0,328.9,6.82,11.6,Completed,3.682000,139.2,512.53


In [ ]:
# =========================
# 5) Only keep relevant columns for modeling
# =========================
loads_small = loads[
    [LOAD_ID, LOAD_ROUTE_ID, LOAD_CUSTOMER_ID, LOAD_REVENUE, LOAD_BOOKING_TYPE, LOAD_WEIGHT_LBS, LOAD_NUMBER_OF_PIECES, LOAD_FUEL_SURCHARGE]
].copy()

routes_small = routes[[ROUTE_ID, ROUTE_DISTANCE_MILES, ROUTE_ORIGIN, ROUTE_DEST, ROUTE_BASE_RATE_PER_MILE, ROUTE_FUEL_SURCHARGE_PER_MILE]].copy()

trips_small = trips_with_fuel[
    [
        TRIP_LOAD_ID,
        TRIP_DURATION_HOURS,
        "avg_fuel_price",
        "total_fuel_quantity",
        "total_fuel_cost",
    ]
].copy()

customers_small = customers[
    [CUSTOMER_ID, CUSTOMER_PRIMARY_FREIGHT_TYPE, CUSTOMER_ANNUAL_REVENUE_POTENTIAL]
].copy()

In [ ]:
# =========================
# 6) JOIN LOADS, ROUTES, TRIPS, CUSTOMERS
# =========================
df = loads_small.merge(
    routes_small, left_on=LOAD_ROUTE_ID, right_on=ROUTE_ID, how="left"
)

df = df.merge(trips_small, left_on=LOAD_ID, right_on=TRIP_LOAD_ID, how="left")

df = df.merge(
    customers_small, left_on=LOAD_CUSTOMER_ID, right_on=CUSTOMER_ID, how="left"
)

In [ ]:
# =========================
# 7) Build simple features
# =========================
df["revenue_per_mile"] = df[LOAD_REVENUE] / df[ROUTE_DISTANCE_MILES]
df["fuel_cost_per_mile"] = df["total_fuel_cost"] / df[ROUTE_DISTANCE_MILES]
df["fuel_quantity_per_mile"] = df["total_fuel_quantity"] / df[ROUTE_DISTANCE_MILES]
df["revenue_per_hour"] = df[LOAD_REVENUE] / df[TRIP_DURATION_HOURS]

# optional: Lane-Feature
df["lane"] = df[ROUTE_ORIGIN].astype(str) + "_" + df[ROUTE_DEST].astype(str)

In [ ]:
# =========================
# 8) cleaning
# =========================
# Remove infinities
df = df.replace([float("inf"), float("-inf")], pd.NA)

# Treat missing values (for simplicity, we drop them here, but you could also impute them)
df = df.dropna(subset=[LOAD_REVENUE, ROUTE_DISTANCE_MILES, TRIP_DURATION_HOURS])

In [ ]:
# =========================
# 9) Drop columns that won't be used for modeling
# =========================
# goal: pricing training table
# target = LOAD_REVENUE
drop_cols = [
    ROUTE_ID,
    TRIP_LOAD_ID,
    FUEL_TRIP_ID if FUEL_TRIP_ID in df.columns else None,
    CUSTOMER_ID if CUSTOMER_ID in df.columns else None,
]
drop_cols = [c for c in drop_cols if c is not None and c in df.columns]

pricing_df = df.drop(columns=drop_cols)


In [ ]:
print(pricing_df.head() )

In [ ]:
# =========================
# 10) Save processed dataset
# =========================
pricing_df.to_csv("processed_datasets/pricing_training_table.csv", index=False)

print(pricing_df.head())
print(pricing_df.shape)

   revenue     load_type  weight_lbs  pieces  fuel_surcharge  \
0  3045.23       Dry Van       19178      13          406.72   
1  1224.48       Dry Van       27761      22           98.61   
2  7171.12  Refrigerated       35594      16          792.88   
3  1308.20  Refrigerated       33274      10          141.33   
4  3317.18       Dry Van       40257      10          738.48   

   typical_distance_miles  origin_city destination_city  base_rate_per_mile  \
0                    1271      Houston          Detroit                2.33   
1                     519  Kansas City     Indianapolis                2.27   
2                    2332     Columbus         Portland                2.69   
3                     673      Phoenix           Denver                1.70   
4                    2172      Houston          Seattle                1.77   

   fuel_surcharge_rate  ...  avg_fuel_price  total_fuel_quantity  \
0                 0.32  ...        3.845000                201.2   
1   